In [1]:
import pandas as pd
import numpy as np
from scipy import stats

In [3]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

**Описание**

1. Проверяем, насколько эффективна кампания:

        H₀: конверсия в ad и psa одинакова; H₁: различается.

2. есть 2 группы:

      ad, которые видели рекламный баннер
      
      psa, видевшие нейтральный баннер
3. Соотношение групп указано 96/4. В описании сказано, что эксперимент рандомизированный, и мы на это полагаемся.
4. α = 0,05
5. главная метрика - доля конвертировавшихся

In [2]:
df = pd.read_csv('/content/drive/MyDrive/marketing_AB.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/marketing_AB.csv'

In [ ]:
df.set_index('Unnamed: 0', inplace = True)
df.index.name = 'index'
display(df)

,user id,test group,converted,total ads,most ads day,most ads hour
index,,,,,,
0,1069124,ad,False,130,Monday,20
1,1119715,ad,False,93,Tuesday,22
2,1144181,ad,False,21,Tuesday,18
3,1435133,ad,False,355,Tuesday,10
4,1015700,ad,False,276,Friday,14
...,...,...,...,...,...,...
588096,1278437,ad,False,1,Tuesday,23
588097,1327975,ad,False,1,Tuesday,23
588098,1038442,ad,False,3,Tuesday,23


In [ ]:
#проверяем что user_id уникальны и что соотношение верное
display(df['user id'].is_unique)
percentages = df['test group'].value_counts(normalize=True) * 100
print(percentages)

True

test group
ad     96.000007
psa     3.999993
Name: proportion, dtype: float64


In [ ]:
#n количество клиентов
n_ad  = (df["test group"] == "ad").sum()
n_psa = (df["test group"] == "psa").sum()
n_total = n_ad + n_psa
print(n_ad,n_psa, n_total   )

564577 23524 588101


In [ ]:
# количество перешедших и конверсия
c  = df.groupby('test group')['converted'].sum()
display(c)
x_ad = c.loc['ad']
x_psa = c.loc['psa']
p_ad = x_ad/ n_ad
p_psa = x_psa/ n_psa
p = (x_ad + x_psa)/ n_total


,converted
test group,
ad,14423
psa,420


In [ ]:
# z и p value
z = (p_ad - p_psa) / np.sqrt(p * (1 - p) * (1 / n_psa + 1 / n_ad))
p_value = 2 * stats.norm.sf(abs(z))
print(z, p_value)

7.3700781265454145 1.7052807161559727e-13


In [ ]:
se = np.sqrt(p_ad * (1 - p_ad) / n_ad + p_psa * (1 - p_psa) / n_psa)
ci = (p_ad - p_psa - 1.96 * se, p_ad - p_psa + 1.96 * se)
print(f"95% Доверительный интервал: [{ci[0]:.4%}, {ci[1]:.4%}]")

95% Доверительный интервал: [0.5951%, 0.9434%]


In [ ]:
print(f"Конверсия тестовая: {p_ad:.4%}")
print(f"Конверсия контрольная: {p_psa:.4%}")
print(f"Разница конверсий: {p_ad - p_psa:.4%}")
print(f'Относительный прирост: {(p_ad/p_psa - 1):.0%}')
print(f'Дополнительные конверсии от рекламы {((p_ad - p_psa) * n_ad):.0f} из {x_ad}')
print(f'Доля, обязанная рекламе {(p_ad - p_psa) / p_ad:.1%}')

Конверсия тестовая: 2.5547%
Конверсия контрольная: 1.7854%
Разница конверсий: 0.7692%
Относительный прирост: 43%
Дополнительные конверсии от рекламы 4343 из 14423
Доля, обязанная рекламе 30.1%


**Вывод**

Конверсия в группе ad (2,55%) выше, чем в psa (1,79%): разница 0,77 п.п.
(95% ДИ: 0,60–0,94 п.п.), относительный прирост 43%.
Разница статистически значима (z = 7,37, p ≈ 1,7·10⁻¹³), H₀ отвергаем.

Реклама дала примерно 4,3 тыс. дополнительных конверсий (около 30% конверсий
группы ad). Остальные ~70% произошли бы и без неё.

Ограничения: рандомизация принята по описанию датасета и не проверялась;
сравнение идёт с PSA, а не с отсутствием показа; в данных нет выручки и
затрат, поэтому денежный эффект не оценён.